In [12]:
REPO_URL = "https://github.com/gwgundersen/transformer.git"
BRANCH = "main"
REPO_DIR = "/content/transformer"
DRIVE_DIR = "/content/drive/MyDrive/transformer"

In [13]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# Clone, or hard-reset an existing clone to the latest remote branch
import os

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

/content/transformer
ff2562c (HEAD -> main, origin/main, origin/HEAD) fix tokenization


In [15]:
# Pin uv to the version the build backend expects (silences the uv_build warning), then build the
# venv with CUDA JAX once
!pip install -q "uv==0.12.18"
!uv sync --frozen --extra cuda
!uv run --no-sync python -c "import jax; print(jax.devices()); assert jax.default_backend() == 'gpu'"

Checked 134 packages in 2ms
[CudaDevice(id=0)]


In [16]:
# Persist artifacts and the HF dataset/tokenization cache on Drive
from pathlib import Path

def symlink(link: str, target: str) -> None:
    link, target = Path(link), Path(target)
    target.mkdir(parents=True, exist_ok=True)
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise FileExistsError(f"{link} is a real directory; move its contents to {target}")
    link.symlink_to(target, target_is_directory=True)
    print(f"{link} -> {link.resolve()}")

symlink(f"{REPO_DIR}/artifacts/vocab",        f"{DRIVE_DIR}/vocab")
symlink(f"{REPO_DIR}/artifacts/checkpoints",  f"{DRIVE_DIR}/checkpoints")
symlink(f"{REPO_DIR}/src/transformer/.cache", f"{DRIVE_DIR}/cache")

/content/transformer/artifacts/vocab -> /content/drive/MyDrive/transformer/vocab
/content/transformer/artifacts/checkpoints -> /content/drive/MyDrive/transformer/checkpoints
/content/transformer/src/transformer/.cache -> /content/drive/MyDrive/transformer/cache


In [17]:
# Build the vocab only if it isn't already on Drive
if all(Path(f"{DRIVE_DIR}/vocab/{lang}.json").exists() for lang in ("de", "en")):
    print("Vocab found on Drive, skipping")
else:
    !uv run --no-sync python -u scripts/generate_iwslt_vocab.py

Vocab found on Drive, skipping


In [ ]:
# Train. Output streams here and is appended to a log on Drive.
!uv run --no-sync python -u scripts/train.py 2>&1 | tee -a {DRIVE_DIR}/train.log

JAX devices: [CudaDevice(id=0)]
Generating train split: 206112 examples [00:00, 571900.22 examples/s]
Generating validation split: 888 examples [00:00, 73114.82 examples/s]
Generating test split: 8079 examples [00:00, 288129.70 examples/s]
Map: 100%|██████████| 206112/206112 [01:07<00:00, 3073.83 examples/s]
0	3.3493
	<s> OK , another , another three - digit number , sir ? </s>
	<s> OK , right ? </s>
1	3.7518
	<s> And I 've always thought it a shame that these giant , plant - eating dinosaurs are too often portrayed as passive , lumbering <unk> of meat on the landscape . </s>
	<s> I always thought , it 's a huge amount of , that these large , often often often often often often , as a huge landscape of the landscape . </s>
2	2.8489
	<s> They went back up , and the Bedouin , each Bedouin , four of them , got behind each of the camels with a little whip . </s>
	<s> And the police -- they were four - year - old -- they were able to get a little bit of a little bit behind a little bit . </